In [1]:
import pandas as pd
import numpy as np
import csv
import os
import time
from glob import glob
import matplotlib.pyplot as plt

## Value Dictionary

In [ ]:
program_path = "/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive"
print(program_path)

In [ ]:
def ObtenerArchivos(ruta_actual,carpeta):
    """Esta función nos devuelve una lista con los archivos de una carpeta"""
    ruta_completa = os.path.join(ruta_actual,carpeta)
    archivos = glob(ruta_completa+"/*")
    return archivos

In [ ]:
files = ObtenerArchivos(program_path,'Dictionaries')
list(files)

In [ ]:
data = {'value_id':['1','2','3','4','5','6','7','8','9','10'],
        'value_short':['SE','CO','TR','BE','UN','SD','ST','HE','AC','PO'],
        'value_long':['SECURITY','CONFORMITY','TRADITION','BENEVOLENCE','UNIVERSALISM',
                      'SELF-DIRECTION','STIMULATION','HEDONISM','ACHIVEMENT','POWER']}

data_df = pd.DataFrame.from_dict(data, dtype='object')
data_df

In [ ]:
lista_txt = []
for archivo in files:
    print("archivo: ",archivo)
    df = pd.read_csv(archivo, sep="\t", names=['word','value_id'])
    df["ID_FILE"] = archivo
    df[['Directorio','Archivo']] = df["ID_FILE"].astype(str).str.split("Dictionaries",expand = True)
    df["dic_type"] = df["Archivo"].astype(str).str.strip("/").str.strip("*\.txt").str.strip("_dictionary")
    df["value_id"] = df["value_id"].astype(str).str.strip("'")
    df.drop(columns=['ID_FILE', 'Directorio', 'Archivo'], inplace=True)
    lista_txt.append(df)
        
dics = pd.concat(lista_txt,axis=0, sort = False)
dics.reset_index(inplace=True,drop=True)

value_dic = dics.merge(data_df, on='value_id')
value_dic.drop_duplicates(subset='word',inplace=True, keep='last', ignore_index=True)

value_dic

## VALUENET (balanced)

### Reading and Concatenating

In [ ]:
program_path = "/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_original"
files = ObtenerArchivos(program_path,'ByValues')
list(files)

In [ ]:
lista_csv = []
for archivo in files:
    print("archivo: ",archivo)
    df = pd.read_csv(archivo, sep=",", header=0)
    df["ID_FILE"] = archivo
    df[["Directorio","Archivo"]] = df["ID_FILE"].astype(str).str.split("ByValues",expand = True)
    df["value"] = df["Archivo"].astype(str).str.strip("/").str.strip("*\.csv")
    df.drop(columns=['Unnamed: 0', 'ID_FILE', 'Directorio', 'Archivo'], inplace=True)
    df = df[['uid','value','scenario','label']]
    lista_csv.append(df)
        
valuenet = pd.concat(lista_csv,axis=0, sort = False)
valuenet.reset_index(inplace=True,drop=True)
valuenet

In [ ]:
valuenet[valuenet['uid']==3055]

In [ ]:
valuenet.groupby('value').count()

In [ ]:
valuenet.groupby('value').count()

In [ ]:
len(valuenet.groupby('scenario').count().sort_values(by='value'))

In [ ]:
valuenet[valuenet['scenario']=="not backing up my mom"]

In [ ]:
#EXPORT
valuenet.to_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all.csv",
#                header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")

In [ ]:
valuenet['label'] = abs(valuenet['label'])
valuenet_pivot = valuenet.pivot(index='scenario', columns='value', values='label').fillna(0).astype(int)
valuenet_pivot.columns.name = None
valuenet_pivot.reset_index(inplace=True)
#valuenet_pivot.rename_axis(None)
valuenet_pivot

In [ ]:
type(valuenet_pivot)

In [ ]:
valuenet_pivot.index

In [ ]:
valuenet_pivot[valuenet_pivot['scenario']=="not backing up my mom"]

In [ ]:
valuenet = pd.melt(valuenet_pivot, id_vars=['scenario'],
                   value_vars=['ACHIEVEMENT', 'BENEVOLENCE', 'CONFORMITY', 'HEDONISM','POWER',
                               'SECURITY', 'SELF-DIRECTION', 'STIMULATION', 'TRADITION','UNIVERSALISM'],
                   ignore_index=True)
valuenet.reset_index(inplace=True)
valuenet

In [ ]:
valuenet.rename(columns={"index": "uid","variable":"value","value":"label"}, inplace=True)
valuenet

In [ ]:
valuenet[valuenet['value']==1]

In [ ]:
valuenet.groupby('variable')['value'].agg(['count','sum'])

In [ ]:
valuenet.groupby('scenario').count()

In [ ]:
valuenet_pivot.columns

In [ ]:
valuenet_pivot.rename(columns=str.upper, inplace=True)
valuenet_pivot

In [ ]:
#EXPORT
valuenet.to_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all_10.csv",
#                header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")

### Train+Test+Eval

In [ ]:
program_path = "/Proyecto/Value-disagreement/Datos/ValueNET"
print(program_path)

In [ ]:
files = ObtenerArchivos(program_path,'v0.3_balanced')
list(files)

In [ ]:
lista_csv = []
for archivo in files:
    if "train" in archivo or "test" in archivo or "eval" in archivo:
        print("archivo: ",archivo)
        df = pd.read_csv(archivo,
                         sep=",",
                         header=0
                         )    
        df["ID_FILE"] = archivo
        df[["Directorio","Archivo"]] = df["ID_FILE"].astype(str).str.split("v0.3_balanced",expand = True)
        df["type"] = df["Archivo"].astype(str).str.strip("/").str.strip("*\.csv").str.upper()

        df['value'] = df['scenario'].str.extract('(\[.+\])', expand=True)
        df["value"] = df["value"].astype(str).str.strip("[").str.strip("]")

        df['scenario'] = df['scenario'].str.extract('(\s.*)', expand=True)
        df["scenario"] = df["scenario"].astype(str).str.strip(" ").str.strip("\t")
        
        df.drop(columns=['ID_FILE', 'Directorio', 'Archivo'], inplace=True)
        df = df[['type','value','scenario','label']]
        lista_csv.append(df)
        
valuenet_dfs = pd.concat(lista_csv,axis=0, sort = False)
valuenet_dfs.reset_index(inplace=True,drop=True)
valuenet_dfs

In [ ]:
len(valuenet_dfs.groupby('scenario').count().sort_values(by='value'))

In [ ]:
valuenet_dfs[valuenet_dfs['scenario']=="not backing up my mom"]

## VALUEARG

### Reading and Concatenating

In [ ]:
program_path = "/Proyecto/Value-disagreement/Datos/ValueARG"
files = ObtenerArchivos(program_path,'webis-argvalues-22')
list(files)

In [ ]:
arguments = pd.read_csv(files[6],
                        sep="\t",
                        header=0
                        )
arguments

In [ ]:
lbl1 = pd.read_csv(files[7],
                        sep="\t",
                        header=0
                        )
lbl1

In [ ]:
lbl2 = pd.read_csv(files[0],
                        sep="\t",
                        header=0
                        )
lbl2

In [ ]:
lbl2[['Self-direction','Power','Security','Conformity','Benevolence','Universalism']] = [0,0,0,0,0,0]
lbl2.loc[(lbl2["Self-direction: thought"] == 1) | (lbl2["Self-direction: action"] == 1), "Self-direction"] = 1
lbl2.loc[(lbl2["Power: dominance"] == 1) | (lbl2["Power: resources"] == 1), "Power"] = 1
lbl2.loc[(lbl2["Security: personal"] == 1) | (lbl2["Security: societal"] == 1), "Security"] = 1
lbl2.loc[(lbl2["Conformity: rules"] == 1) | (lbl2["Conformity: interpersonal"] == 1), "Conformity"] = 1
lbl2.loc[(lbl2["Benevolence: caring"] == 1) | (lbl2["Benevolence: dependability"] == 1), "Benevolence"] = 1
lbl2.loc[(lbl2["Universalism: nature"] == 1) | (lbl2["Universalism: concern"] == 1) | (lbl2["Universalism: tolerance"] == 1) | (lbl2["Universalism: objectivity"] == 1), "Universalism"] = 1
lbl2 = lbl2[['Argument ID', 'Security', 'Conformity', 'Tradition', 'Benevolence', 
             'Universalism', 'Self-direction', 'Stimulation', 'Hedonism', 'Achievement', 'Power']]
lbl2

In [ ]:
lbl2 = lbl2.merge(arguments[['Argument ID','Premise']], on='Argument ID')
#lbl2['variable'] = lbl2['variable'].str.upper()
lbl2.rename(columns={"Premise":"Scenario"}, inplace=True)
lbl2

In [ ]:
#EXPORT
lbl2.to_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all_10.csv",
#            header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")

In [ ]:
lbl2 = pd.melt(lbl2, id_vars=['Argument ID'],
               value_vars=['Security','Conformity','Tradition','Benevolence','Universalism',
                           'Self-direction','Stimulation','Hedonism','Achievement', 'Power'],
               ignore_index=False)
lbl2

In [ ]:
lbl2 = lbl2.merge(arguments[['Argument ID','Premise']], on='Argument ID')
lbl2['variable'] = lbl2['variable'].str.upper()
lbl2

In [ ]:
lbl2.groupby('variable').count()

In [ ]:
lbl2.rename(columns={"Premise":"Scenario"}, inplace=True)
lbl2 = lbl2[['Scenario','Achievement','Benevolence','Conformity','Hedonism',
             'Power','Security','Self-direction','Stimulation','Tradition','Universalism']]
lbl2.rename(columns=str.upper, inplace=True)
lbl2

In [ ]:
#EXPORT
lbl2.to_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all.csv",
#            header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")

## VALUE ALL

In [ ]:
lbl2.rename(columns={"Argument ID": "uid","variable":"value","value":"label","Premise":"scenario"}, inplace=True)
lbl2.drop_duplicates(subset=['value','scenario'],inplace=True, keep='first', ignore_index=True)
lbl2

In [ ]:
value_all = pd.concat([valuenet,lbl2],axis=0, sort = False) ## 2733 rows
value_all

In [ ]:
value_all.reset_index(drop= True ,inplace= True)
value_all

In [ ]:
len(value_all)

In [ ]:
value_all.groupby('value').count()

In [ ]:
value_all.to_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv",
            header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")

In [ ]:
value_all.to_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_10.csv",
#            header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")